In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)



In [ ]:
# Task 2: Write your code here:
df_Food.head()


In [ ]:
# Task 3: Write your code here:
print("DataFrame Info:")
df_food.info()

In [ ]:
# Task 4: Write your code here:
print("\nDataFrame Description:")
df_food.describe()

In [ ]:
# Task 5: Write your code here:
# 1. What does our target variable (delivery_time) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_food, "Delivery_Time")

In [ ]:
df = df_food.drop(["Order_ID"], axis=1)

In [ ]:
# Check the  data again types and structure
df_food.info()

In [ ]:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_food)

In [ ]:
# Task 3: Write your code here:

feature_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']
target_col = 'Delivery_Time'


df_clean = df_food.dropna(subset=feature_cols + [target_col]).copy()

# Split into X and y
X = df_clean[feature_cols]
y = df_clean[target_col]

print(f"Shape after cleaning: {df_clean.shape}")
print("Missing values in y:", y.isna().sum())
print("X shape:", X.shape, "y shape:", y.shape)


In [ ]:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_food)

In [ ]:
# 3. Do we have categorical columns?
categorical_cols = df_food.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_food[col] = le.fit_transform(df_food[col])
  label_encoders[col] = le

df_food

In [ ]:
# check if Do we have different scales in the data?
df_food.describe()

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_food.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_food[numerical_cols] = scaler.fit_transform(df_food[numerical_cols])
df_food.head()


In [ ]:
# 1. Is the target imbalanced?
import seaborn as sns
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_food, "Delivery_Time")

In [ ]:

df_clean = df_food.dropna().copy()


X = df_clean.drop("Delivery_Time", axis=1)
y = df_clean["Delivery_Time"]

# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


print("NaNs in y_train:", y_train.isna().sum())
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")


In [ ]:

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train):

    X_fold_train = X_train.iloc[train_idx]
    X_fold_val   = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val   = y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print("5-Fold CV Results:")

print(f"Average MAE across all folds: {mae_scores.mean():.2f}")


In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Delivery_Time distribution
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black', color='green')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
